# Deep Learning – Multiclass Intrusion Detection

This notebook applies a Deep Learning model to classify IoT network traffic
into multiple classes (normal traffic and different attack types).

The model is trained on a balanced dataset generated from IoT-Flock traffic.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import ipaddress

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns


## 2. Loading the Dataset

In [ ]:
df = pd.read_csv("balanced_dataset.csv")
df.head()


## 3. IP Address Conversion

In [ ]:
def ip_to_int(ip):
    try:
        return int(ipaddress.ip_address(ip))
    except:
        return 0

df['ip.src'] = df['ip.src'].apply(ip_to_int)
df['ip.dst'] = df['ip.dst'].apply(ip_to_int)


## 4. Checking for Messing Values

In [ ]:
df.isnull().sum()


## 5. Label Encoding (MultiClass)

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['label'])
y_cat = to_categorical(y)

label_encoder.classes_


## 6. Standardization of features

In [ ]:
X = df.drop(columns=['label'])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 7. Train Separation / Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_cat,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train.shape, X_test.shape


## 8. Deep Learning Model architecture

In [ ]:
model = Sequential()

model.add(Dense(128, activation='relu', input_shape=(X_train.shape[1],)))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.3))

model.add(Dense(y_train.shape[1], activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


## 9. Model Training

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stop]
)


## 10. Evaluation (Classification Report)

In [ ]:
y_pred = model.predict(X_test)

y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

print(classification_report(
    y_true,
    y_pred_classes,
    target_names=label_encoder.classes_
))


## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix – Deep Learning')
plt.tight_layout()
plt.show()


## 12. Learning Curves

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Model Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Model Loss')
plt.legend()

plt.tight_layout()
plt.show()


## Conclusion

The deep learning model demonstrates strong performance for multiclass
intrusion detection on IoT traffic. Batch normalization and dropout
helped reduce overfitting, while early stopping improved generalization.